In [1]:
import pandas as pd

# Load files
PartsMovement_df = pd.read_excel("Parts Movement agc 19 Okt 25.xlsx")
Forecast_df      = pd.read_excel("FD agc 19 okt 2025.xlsx")

# --- STEP 2: Identify all C- columns ---
c_columns = sorted(
    [col for col in PartsMovement_df if isinstance(col, str) and col.startswith("C-")],
    key=lambda x: int(x.split("-")[1]),
    reverse=True
)

# Required column names
oh_col = "OH"
oo_col = "OO"
dn_price_col = "DN Price"

# --- STEP 3: SUM OH, OO, and all C- columns ---
sum_columns = c_columns + [oh_col, oo_col]

# Aggregation rules
aggregation_dict = {col: "sum" for col in sum_columns}
aggregation_dict[dn_price_col] = "first"   # keep 1 DN Price per PN

# --- STEP 4: Group at national level (remove Brc + Agc later) ---
df_sum = PartsMovement_df.groupby(["Agc", "P/N"], as_index=False).agg(aggregation_dict)

# --- STEP 5: Create Total Calls ---
df_sum["Total Calls"] = df_sum[c_columns].sum(axis=1)

# --- STEP 6: Remove individual C- columns ---
df_sum = df_sum.drop(columns=c_columns)

# --- STEP 7: Remove Brc and Agc as requested ---
df_sum = df_sum.drop(columns=["Agc"])

# --- STEP 8: Reorder columns ---
df_sum = df_sum[["P/N", "Total Calls", "DN Price", "OH", "OO"]]

# --- STEP 9: Sort by Total Calls descending ---
df_sum = df_sum.sort_values(by="Total Calls", ascending=False)

# Show final result
print(df_sum)


           P/N  Total Calls  DN Price      OH      OO
2777   3164067        856.0     17.08   647.0   138.0
109     129839        304.0      2.46   248.0    37.0
385     193736        296.0      2.32  1284.0  2333.0
579     207244        276.0      4.91   957.0   366.0
4758   3803615        265.0     23.94    91.0     6.0
...        ...          ...       ...     ...     ...
6142   3975487          0.0     14.10     9.0     0.0
6150   3976063          0.0    103.67     0.0     2.0
1065   3008840          0.0   1278.85     1.0     0.0
6156   3976644          0.0     19.17     1.0     0.0
10021  s   696          0.0      5.52     4.0     0.0

[10022 rows x 5 columns]


In [2]:
#Perhitungan RC (Rank Call)
# Sort by Total Calls descending
df_sum = df_sum.sort_values(by="Total Calls", ascending=False).reset_index(drop=True)

# Add cumulative sum
df_sum["Accum."] = df_sum["Total Calls"].cumsum()

# Percentage cumulative
last_accum = df_sum["Accum."].iloc[-1]
df_sum["%Accum."] = (df_sum["Accum."] / last_accum) * 100
df_sum["%Accum."] = df_sum["%Accum."].round(2)

#klasifikasi RC ABCD
def assign_rc(pct):
    if pct < 50:
        return "A"
    elif pct < 80:
        return "B"
    elif pct < 100:
        return "C"
    else:   # pct == 100
        return "D"


df_sum["RC"] = df_sum["%Accum."].apply(assign_rc)

print(df_sum)


           P/N  Total Calls  DN Price      OH      OO   Accum.  %Accum. RC
0      3164067        856.0     17.08   647.0   138.0    856.0     1.50  A
1       129839        304.0      2.46   248.0    37.0   1160.0     2.03  A
2       193736        296.0      2.32  1284.0  2333.0   1456.0     2.55  A
3       207244        276.0      4.91   957.0   366.0   1732.0     3.03  A
4      3803615        265.0     23.94    91.0     6.0   1997.0     3.49  A
...        ...          ...       ...     ...     ...      ...      ... ..
10017  3649992          0.0     40.01     1.0     0.0  57178.0   100.00  D
10018  3045046          0.0     10.39     8.0     0.0  57178.0   100.00  D
10019  3045045          0.0      5.21     0.0     0.0  57178.0   100.00  D
10020  3045039          0.0      5.04     2.0     0.0  57178.0   100.00  D
10021  s   696          0.0      5.52     4.0     0.0  57178.0   100.00  D

[10022 rows x 8 columns]


In [3]:
# --- STEP 1: Make sure both DataFrames use the same part number column --- 
# If Forecast_df uses "PN", rename it to match "P/N"
if "p/n" in Forecast_df.columns:
    Forecast_df = Forecast_df.rename(columns={"p/n": "P/N"})

# Keep only P/N and FD_final
forecast_subset = Forecast_df[["P/N", "FD_final"]]

# --- STEP 2: Merge into df_sum ---
df_sum = df_sum.merge(forecast_subset, on="P/N", how="left")

# --- STEP 3: Move FD_final to last column (optional) ---
fd_val = df_sum.pop("FD_final")
df_sum["FD_final"] = fd_val

# --- Show result ---
print(df_sum)


           P/N  Total Calls  DN Price      OH      OO   Accum.  %Accum. RC  \
0      3164067        856.0     17.08   647.0   138.0    856.0     1.50  A   
1       129839        304.0      2.46   248.0    37.0   1160.0     2.03  A   
2       193736        296.0      2.32  1284.0  2333.0   1456.0     2.55  A   
3       207244        276.0      4.91   957.0   366.0   1732.0     3.03  A   
4      3803615        265.0     23.94    91.0     6.0   1997.0     3.49  A   
...        ...          ...       ...     ...     ...      ...      ... ..   
10017  3649992          0.0     40.01     1.0     0.0  57178.0   100.00  D   
10018  3045046          0.0     10.39     8.0     0.0  57178.0   100.00  D   
10019  3045045          0.0      5.21     0.0     0.0  57178.0   100.00  D   
10020  3045039          0.0      5.04     2.0     0.0  57178.0   100.00  D   
10021  s   696          0.0      5.52     4.0     0.0  57178.0   100.00  D   

       FD_final  
0         153.0  
1          13.0  
2        

In [4]:
#Urutan Kolom df_sum
# --- Define your desired final column order ---
final_columns = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "OH",
    "OO",
    "Accum.",
    "%Accum."
]

# --- Reorder df_sum (only keep those that exist) ---
df_sum = df_sum[final_columns]

print(df_sum)

           P/N  Total Calls RC  DN Price  FD_final      OH      OO   Accum.  \
0      3164067        856.0  A     17.08     153.0   647.0   138.0    856.0   
1       129839        304.0  A      2.46      13.0   248.0    37.0   1160.0   
2       193736        296.0  A      2.32     774.0  1284.0  2333.0   1456.0   
3       207244        276.0  A      4.91     299.0   957.0   366.0   1732.0   
4      3803615        265.0  A     23.94      24.0    91.0     6.0   1997.0   
...        ...          ... ..       ...       ...     ...     ...      ...   
10017  3649992          0.0  D     40.01       0.0     1.0     0.0  57178.0   
10018  3045046          0.0  D     10.39       0.0     8.0     0.0  57178.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0  57178.0   
10020  3045039          0.0  D      5.04       0.0     2.0     0.0  57178.0   
10021  s   696          0.0  D      5.52       NaN     4.0     0.0  57178.0   

       %Accum.  
0         1.50  
1         2.03  


In [5]:
#Perhitungan Max
# RC → multiplier
rc_multiplier = {
    "A": 7,
    "B": 5.5,
    "C": 3,
    "D": 0
}

# Convert RC class into numeric multiplier
df_sum["RC_value"] = df_sum["RC"].map(rc_multiplier)
df_sum["Max"] = df_sum["FD_final"] * df_sum["RC_value"]
df_sum["Max"] = df_sum["Max"].round(2)
df_sum = df_sum.drop(columns=["RC_value"])

#Urutan Kolom df_sum
# --- Define your desired final column order ---
final_columns = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "Max",
    "OH",
    "OO",
    "Accum.",
    "%Accum."
]

# --- Reorder df_sum (only keep those that exist) ---
df_sum = df_sum[final_columns]

print(df_sum)

           P/N  Total Calls RC  DN Price  FD_final     Max      OH      OO  \
0      3164067        856.0  A     17.08     153.0  1071.0   647.0   138.0   
1       129839        304.0  A      2.46      13.0    91.0   248.0    37.0   
2       193736        296.0  A      2.32     774.0  5418.0  1284.0  2333.0   
3       207244        276.0  A      4.91     299.0  2093.0   957.0   366.0   
4      3803615        265.0  A     23.94      24.0   168.0    91.0     6.0   
...        ...          ... ..       ...       ...     ...     ...     ...   
10017  3649992          0.0  D     40.01       0.0     0.0     1.0     0.0   
10018  3045046          0.0  D     10.39       0.0     0.0     8.0     0.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0     0.0   
10020  3045039          0.0  D      5.04       0.0     0.0     2.0     0.0   
10021  s   696          0.0  D      5.52       NaN     NaN     4.0     0.0   

        Accum.  %Accum.  
0        856.0     1.50  
1       116

In [6]:
#INCOMING
# List of new incoming columns
incoming_cols = [f"Incoming M-{i}" for i in range(1, 8)]

# Insert after OO
oo_index = df_sum.columns.get_loc("OO")

for i, col_name in enumerate(incoming_cols):
    if col_name not in df_sum.columns:        # <-- Prevent duplicate error
        df_sum.insert(oo_index + 1 + i, col_name, "")
        
print(df_sum)


           P/N  Total Calls RC  DN Price  FD_final     Max      OH      OO  \
0      3164067        856.0  A     17.08     153.0  1071.0   647.0   138.0   
1       129839        304.0  A      2.46      13.0    91.0   248.0    37.0   
2       193736        296.0  A      2.32     774.0  5418.0  1284.0  2333.0   
3       207244        276.0  A      4.91     299.0  2093.0   957.0   366.0   
4      3803615        265.0  A     23.94      24.0   168.0    91.0     6.0   
...        ...          ... ..       ...       ...     ...     ...     ...   
10017  3649992          0.0  D     40.01       0.0     0.0     1.0     0.0   
10018  3045046          0.0  D     10.39       0.0     0.0     8.0     0.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0     0.0   
10020  3045039          0.0  D      5.04       0.0     0.0     2.0     0.0   
10021  s   696          0.0  D      5.52       NaN     NaN     4.0     0.0   

      Incoming M-1 Incoming M-2 Incoming M-3 Incoming M-4 Incom

In [7]:
#Estimated OH & Estimated OO
import numpy as np

# ----- CREATE COLUMN NAMES -----
est_oh_cols = [f"Estimated OH M-{i}" for i in range(1, 10)]   # M-1 to M-9
est_oo_cols = [f"Estimated OO M-{i}" for i in range(1, 9)]    # M-1 to M-8

# Insert Estimated OH columns after Incoming M-7
insert_pos = df_sum.columns.get_loc("Incoming M-7") + 1
for i, col in enumerate(est_oh_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# Insert Estimated OO columns after Estimated OH M-9
insert_pos = df_sum.columns.get_loc("Estimated OH M-9") + 1
for i, col in enumerate(est_oo_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE VALUES ROW-BY-ROW -----
for idx, row in df_sum.iterrows():

    # Get fixed values
    OH0 = row["OH"]
    OO0 = row["OO"]
    FD = row["FD_final"]
    Max = row["Max"]

    # Store results
    est_oh = {}
    est_oo = {}

    # ----- Estimated OH M-1 -----
    incoming_1 = float(row["Incoming M-1"]) if row["Incoming M-1"] not in ["", None, np.nan] else 0
    est_oh[1] = (OH0 + OO0 + incoming_1) - FD

    # ----- Estimated OO M-1 -----
    est_oo[1] = 0 if est_oh[1] > Max else (Max - est_oh[1])

    # ----- M-2 to M-9 for Estimated OH, M-2 to M-8 for Estimated OO -----
    for i in range(2, 10):

        incoming_i = (
            float(row[f"Incoming M-{i}"])
            if (f"Incoming M-{i}" in df_sum.columns and row[f"Incoming M-{i}"] not in ["", None, np.nan])
            else 0
        )

        # Estimated OH M-i
        prev_oh = est_oh[i-1]
        prev_oo = est_oo[i-1] if i-1 in est_oo else 0
        est_oh[i] = (prev_oh + incoming_i + prev_oo) - FD

        # Estimated OO only until M-8
        if i <= 8:
            est_oo[i] = 0 if est_oh[i] > Max else (Max - est_oh[i])

    # ----- Assign to dataframe -----
    for i in range(1, 10):
        df_sum.loc[idx, f"Estimated OH M-{i}"] = est_oh[i]

    for i in range(1, 9):
        df_sum.loc[idx, f"Estimated OO M-{i}"] = est_oo[i]
        
print(df_sum)


           P/N  Total Calls RC  DN Price  FD_final     Max      OH      OO  \
0      3164067        856.0  A     17.08     153.0  1071.0   647.0   138.0   
1       129839        304.0  A      2.46      13.0    91.0   248.0    37.0   
2       193736        296.0  A      2.32     774.0  5418.0  1284.0  2333.0   
3       207244        276.0  A      4.91     299.0  2093.0   957.0   366.0   
4      3803615        265.0  A     23.94      24.0   168.0    91.0     6.0   
...        ...          ... ..       ...       ...     ...     ...     ...   
10017  3649992          0.0  D     40.01       0.0     0.0     1.0     0.0   
10018  3045046          0.0  D     10.39       0.0     0.0     8.0     0.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0     0.0   
10020  3045039          0.0  D      5.04       0.0     0.0     2.0     0.0   
10021  s   696          0.0  D      5.52       NaN     NaN     4.0     0.0   

      Incoming M-1 Incoming M-2  ... Estimated OO M-1 Estimated

In [8]:
#Schedule Order
import numpy as np

# ----- CREATE COLUMN NAMES -----
schedule_cols = [f"Schedule Order M-{i}" for i in range(1, 7)]

# Insert Schedule Order columns after Estimated OO M-8 (the last OO column)
insert_pos = df_sum.columns.get_loc("Estimated OO M-8") + 1
for i, col in enumerate(schedule_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE SCHEDULE ORDER -----
for idx, row in df_sum.iterrows():

    Max = row["Max"]

    for i in range(1, 7):

        # Lookahead month: OH M-(i+3)
        oh_col = f"Estimated OH M-{i+3}"

        # If OH value exists, fetch it, else use 0
        oh_value = row[oh_col] if oh_col in df_sum.columns else 0

        # Schedule Order formula
        if oh_value > Max:
            sched = 0
        else:
            sched = Max - oh_value

        df_sum.loc[idx, f"Schedule Order M-{i}"] = sched
print(df_sum)

           P/N  Total Calls RC  DN Price  FD_final     Max      OH      OO  \
0      3164067        856.0  A     17.08     153.0  1071.0   647.0   138.0   
1       129839        304.0  A      2.46      13.0    91.0   248.0    37.0   
2       193736        296.0  A      2.32     774.0  5418.0  1284.0  2333.0   
3       207244        276.0  A      4.91     299.0  2093.0   957.0   366.0   
4      3803615        265.0  A     23.94      24.0   168.0    91.0     6.0   
...        ...          ... ..       ...       ...     ...     ...     ...   
10017  3649992          0.0  D     40.01       0.0     0.0     1.0     0.0   
10018  3045046          0.0  D     10.39       0.0     0.0     8.0     0.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0     0.0   
10020  3045039          0.0  D      5.04       0.0     0.0     2.0     0.0   
10021  s   696          0.0  D      5.52       NaN     NaN     4.0     0.0   

      Incoming M-1 Incoming M-2  ... Estimated OO M-7 Estimated

In [9]:
#Amount Schedule Order
import numpy as np

# ----- CREATE COLUMN NAMES -----
amount_cols = [f"Amount Schedule Order M-{i}" for i in range(1, 7)]

# Find the position after "Schedule Order M-6"
insert_pos = df_sum.columns.get_loc("Schedule Order M-6") + 1

# Insert empty columns first (if not already present)
for i, col in enumerate(amount_cols):
    if col not in df_sum.columns:
        df_sum.insert(insert_pos + i, col, np.nan)

# ----- CALCULATE AMOUNTS -----
for idx, row in df_sum.iterrows():

    dn_price = row["DN Price"]

    for i in range(1, 7):

        sched_col = f"Schedule Order M-{i}"
        amount_col = f"Amount Schedule Order M-{i}"

        schedule_value = row[sched_col] if sched_col in df_sum.columns else 0

        df_sum.loc[idx, amount_col] = schedule_value * dn_price
        # Round ONLY the Amount Schedule Order columns
for col in amount_cols:
    df_sum[col] = df_sum[col].round(2)
print(df_sum)

           P/N  Total Calls RC  DN Price  FD_final     Max      OH      OO  \
0      3164067        856.0  A     17.08     153.0  1071.0   647.0   138.0   
1       129839        304.0  A      2.46      13.0    91.0   248.0    37.0   
2       193736        296.0  A      2.32     774.0  5418.0  1284.0  2333.0   
3       207244        276.0  A      4.91     299.0  2093.0   957.0   366.0   
4      3803615        265.0  A     23.94      24.0   168.0    91.0     6.0   
...        ...          ... ..       ...       ...     ...     ...     ...   
10017  3649992          0.0  D     40.01       0.0     0.0     1.0     0.0   
10018  3045046          0.0  D     10.39       0.0     0.0     8.0     0.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0     0.0   
10020  3045039          0.0  D      5.04       0.0     0.0     2.0     0.0   
10021  s   696          0.0  D      5.52       NaN     NaN     4.0     0.0   

      Incoming M-1 Incoming M-2  ... Schedule Order M-5 Schedul

In [10]:
#Description
# --- STEP 1: Extract P/N + Desc from PartsMovement_df ---
desc_df = PartsMovement_df[["P/N", "Desc"]].drop_duplicates()

# --- STEP 2: Merge Desc into df_sum ---
df_sum = df_sum.merge(desc_df, on="P/N", how="left")

# --- STEP 3: Insert Desc after Amount Schedule Order M-6 ---
insert_pos = df_sum.columns.get_loc("Amount Schedule Order M-6") + 1

desc_series = df_sum.pop("Desc")
df_sum.insert(insert_pos, "Desc", desc_series)

print(df_sum)

           P/N  Total Calls RC  DN Price  FD_final     Max      OH      OO  \
0      3164067        856.0  A     17.08     153.0  1071.0   647.0   138.0   
1       129839        304.0  A      2.46      13.0    91.0   248.0    37.0   
2       193736        296.0  A      2.32     774.0  5418.0  1284.0  2333.0   
3       207244        276.0  A      4.91     299.0  2093.0   957.0   366.0   
4      3803615        265.0  A     23.94      24.0   168.0    91.0     6.0   
...        ...          ... ..       ...       ...     ...     ...     ...   
10017  3649992          0.0  D     40.01       0.0     0.0     1.0     0.0   
10018  3045046          0.0  D     10.39       0.0     0.0     8.0     0.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0     0.0   
10020  3045039          0.0  D      5.04       0.0     0.0     2.0     0.0   
10021  s   696          0.0  D      5.52       NaN     NaN     4.0     0.0   

      Incoming M-1 Incoming M-2  ... Schedule Order M-6  \
0   

In [11]:
#MM06
c06_cols = [f"C-{i}" for i in range(1, 7)]  # C-1 to C-6
c06_cols = [col for col in c06_cols if col in PartsMovement_df.columns]  # verify exist
# Group by P/N and sum first
temp = PartsMovement_df.groupby("P/N", as_index=False)[c06_cols].sum()

# Count how many months have calls > 0
temp["MM06"] = temp[c06_cols].gt(0).sum(axis=1)

# Keep only P/N + MM06
mm06_df = temp[["P/N", "MM06"]]
df_sum = df_sum.merge(mm06_df, on="P/N", how="left")
insert_pos = df_sum.columns.get_loc("Total Calls") + 1
mm06_series = df_sum.pop("MM06")
df_sum.insert(insert_pos, "MM06", mm06_series)



In [12]:
#MM12
c12_cols = [f"C-{i}" for i in range(1, 13)]  # C-1 to C-12
c12_cols = [col for col in c12_cols if col in PartsMovement_df.columns]  # ensure exist
temp12 = PartsMovement_df.groupby("P/N", as_index=False)[c12_cols].sum()

# Count how many months have calls > 0 (non-zero values)
temp12["MM12"] = temp12[c12_cols].gt(0).sum(axis=1)

# Keep only P/N + MM12
mm12_df = temp12[["P/N", "MM12"]]
df_sum = df_sum.merge(mm12_df, on="P/N", how="left")
insert_pos = df_sum.columns.get_loc("MM06") + 1
mm12_series = df_sum.pop("MM12")
df_sum.insert(insert_pos, "MM12", mm12_series)

In [ ]:
#Trend Filtered


           P/N  Total Calls  MM06  MM12 RC  DN Price  FD_final     Max  \
0      3164067        856.0     6    12  A     17.08     153.0  1071.0   
1       129839        304.0     6    12  A      2.46      13.0    91.0   
2       193736        296.0     6    12  A      2.32     774.0  5418.0   
3       207244        276.0     6    12  A      4.91     299.0  2093.0   
4      3803615        265.0     6    12  A     23.94      24.0   168.0   
...        ...          ...   ...   ... ..       ...       ...     ...   
10017  3649992          0.0     0     0  D     40.01       0.0     0.0   
10018  3045046          0.0     0     0  D     10.39       0.0     0.0   
10019  3045045          0.0     0     0  D      5.21       0.0     0.0   
10020  3045039          0.0     0     0  D      5.04       0.0     0.0   
10021  s   696          0.0     0     0  D      5.52       NaN     NaN   

           OH      OO  ... Schedule Order M-6 Amount Schedule Order M-1  \
0       647.0   138.0  ...          

In [16]:
#Amount Estimated OO
est_oo_cols = [col for col in df_sum.columns if col.startswith("Estimated OO M-")]
est_oo_cols = sorted(
    est_oo_cols, 
    key=lambda x: int(x.split("M-")[1])  # ensure correct order M-1, M-2, ...
)
# Create monetary columns: DN Price × Estimated OO M-i
for col in est_oo_cols:
    month_num = col.split("M-")[1]  # extracts "1", "2", ... "8"
    new_col = f"Amount Estimated OO M-{month_num}"
    df_sum[new_col] = df_sum["DN Price"] * df_sum[col]
# Find insertion point (after MM12)
insert_pos = df_sum.columns.get_loc("MM12") + 1

# Collect new amount columns in the same order
est_oo_amount_cols = [f"Amount Estimated OO M-{i}" for i in range(1, 9)]

# Move them into correct position
for i, col in enumerate(est_oo_amount_cols):
    series = df_sum.pop(col)
    df_sum.insert(insert_pos + i, col, series)



In [19]:
# --- RENAME COLUMNS FIRST ---

# Create a rename dictionary
rename_dict = {}

for col in df_sum.columns:
    new_col = col
    new_col = new_col.replace("Estimated", "Est.")
    new_col = new_col.replace("Amount", "Amt.")
    new_col = new_col.replace("Schedule","Sched.")
    rename_dict[col] = new_col

# Apply renaming
df_sum = df_sum.rename(columns=rename_dict)


# --- NOW REBUILD THE COLUMN ORDER USING UPDATED NAMES ---

# STARTING COLUMNS
desired_order_start = [
    "P/N",
    "Total Calls",
    "RC",
    "DN Price",
    "FD_final",
    "Max",
    "OH",
    "OO"
]

# Dynamic column groups (renamed ones)
incoming_cols = [col for col in df_sum.columns if col.startswith("Incoming M-")]
est_oh_cols   = [col for col in df_sum.columns if col.startswith("Est. OH M-")]
est_oo_cols   = [col for col in df_sum.columns if col.startswith("Est. OO M-")]
schedule_cols = [col for col in df_sum.columns if col.startswith("Sched. Order M-")]
amt_cols      = [col for col in df_sum.columns if col.startswith("Amt. Sched. Order M-") or col.startswith("Amt. Est. OO")]

ending_cols = [
    "Desc",
    "MM06",
    "MM12",
    "Accum.",
    "%Accum."
]

# Final order (only include columns that exist)
final_order = (
    desired_order_start
    + incoming_cols
    + est_oh_cols
    + est_oo_cols
    + schedule_cols
    + amt_cols
    + ending_cols
)

final_order = [col for col in final_order if col in df_sum.columns]

# Apply reordering
df_sum = df_sum[final_order]

print(df_sum)


           P/N  Total Calls RC  DN Price  FD_final     Max      OH      OO  \
0      3164067        856.0  A     17.08     153.0  1071.0   647.0   138.0   
1       129839        304.0  A      2.46      13.0    91.0   248.0    37.0   
2       193736        296.0  A      2.32     774.0  5418.0  1284.0  2333.0   
3       207244        276.0  A      4.91     299.0  2093.0   957.0   366.0   
4      3803615        265.0  A     23.94      24.0   168.0    91.0     6.0   
...        ...          ... ..       ...       ...     ...     ...     ...   
10017  3649992          0.0  D     40.01       0.0     0.0     1.0     0.0   
10018  3045046          0.0  D     10.39       0.0     0.0     8.0     0.0   
10019  3045045          0.0  D      5.21       0.0     0.0     0.0     0.0   
10020  3045039          0.0  D      5.04       0.0     0.0     2.0     0.0   
10021  s   696          0.0  D      5.52       NaN     NaN     4.0     0.0   

      Incoming M-1 Incoming M-2  ... Amt. Est. OO M-4 Amt. Est.

In [20]:
#OUTPUT
# Export df_sum to Excel
output_path = "FD_Processed_Output.xlsx"

df_sum.to_excel(output_path, index=False)

print(f"File exported successfully: {output_path}")


File exported successfully: FD_Processed_Output.xlsx
